[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NadavFireman/Representation-Learning-Midterm-DAE-SimCLR/blob/main/colab_final.ipynb)

# Part A: the final runs

Every run of Part A, from scratch, into `midterm_runs` on Drive, next to the earlier runs, which are not changed. Its encoders go to `midterm_runs/encoders`.

Every run that trains also scores the test set after every epoch (`epoch/test_*` in TensorBoard, next to `epoch/train_*` and `epoch/val_*`), with every metric: top-1 and top-5 accuracy, macro precision, recall, F1 and AUROC, and k-NN for the encoders. Only val chooses the checkpoint and stops a run; the test curve is never used to choose. Every classifier and every `analysis` run also saves its test confusion matrix.

Every run gets a folder `date<DDMMYYYY>_time<HHMMSS>_<model>_final_<stage>_<what>`, e.g. `date23112026_time101500_ResNet18_final_finetune_DAE_on_TinyImageNet_10pct`, and a `run_info.txt` (also in TensorBoard's TEXT tab): what it is, its hyperparameters, epochs, results, start and end time and training time. A start and an end line with the date and time are printed for every run.

The groups:

1. **CNN-3** (the course's small encoder): the DAE and the AE (12 epochs), then the full protocol. About 5–6 hours.
2. **ResNet-18, full fine-tuning**: (a) the DAE and the AE with the ResNet-18 decoder (30 epochs), the DAE with the light decoder (12 epochs); (b) the full protocol. About 12–14 hours.
3. **ResNet-18, partial fine-tuning**: the encoders of group 2; conv1, layer1 and layer2 stay frozen, layer3, layer4, fc and the head train (`--finetune_from layer3`). About 3 hours.
4. **Seeds 1 and 2** of group 1, for mean ± std. About 5 hours per seed.

Run `colab_tuning.ipynb` first: every fine-tuning, linear probe and supervised run here uses its choice (`tuning_choice.json`). If the tuning is run again after this notebook, first move the `date*_final_*` folders and `encoders` out of `midterm_runs`: a finished run is never trained again, so it would keep the results of the old choice. On two L4 runtimes: groups 1 and 4 in one, groups 2 and 3 in the other. Run *Setup* and *Drive* in both. After a disconnect, run the same cells again: finished runs are skipped, unfinished ones continue from their `last.ckpt`, in the same folder.

## Setup
Downloads the repository (or pulls the latest code), installs it and checks the GPU. Use an L4 or A100 runtime.

In [ ]:
import os

REPO = "/content/Representation-Learning-Midterm-DAE-SimCLR"
if not os.path.exists(REPO):
    !git clone https://github.com/NadavFireman/Representation-Learning-Midterm-DAE-SimCLR.git {REPO}
os.chdir(REPO)
!git pull
!git log --oneline -1

In [ ]:
%pip install -q -r requirements.txt

import torch
assert torch.cuda.is_available(), "No GPU. Runtime → Change runtime type → L4 GPU, then rerun this cell."
print(torch.cuda.get_device_name(0), "|", os.cpu_count(), "CPU cores")

## Drive
`OUT` is where every run writes, next to the earlier runs.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/לימודים/תואר שני/שנה ב/סמסטר א/למידת ייצוגים במודלים עמוקים/מטלות/מטלת אמצע"
OUT = f"{DRIVE_DIR}/midterm_runs"
assert os.path.isdir(OUT), f"no folder {OUT}"
print("writes to:", OUT)

## TensorBoard
All the runs, the earlier ones too. In the run filter on the left, part of a path picks a group, for example `final`, `CNN3_final_pretrain`, `ResNet18_final_finetune` or `from-layer3`, then *Toggle all runs* under the list (with many runs TensorBoard selects none). The `epoch` section has train, val and test on the same epoch axis; the TEXT tab has each run's `run_info`.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir "{OUT}"

## The runs
`run_all` runs train.py once per command. A run that finished (it has `results.json`) prints its results and is skipped.

In [ ]:
import json
import re

FRACTIONS = (0.01, 0.1, 1.0)
DATASETS = ("tinyimagenet", "cifar100")


def tuned(setting):
    """The flags colab_tuning.ipynb chose for setting (e.g. "cnn3_finetune")."""
    path = f"{OUT}/tuning_choice.json"
    assert os.path.exists(path), "run colab_tuning.ipynb first"
    with open(path) as f:
        choice = json.load(f)[setting]
    return "".join(f" --{k} {v}" for k, v in choice.items())


def activation(setting):
    """The --activation of a supervised setting, to load its encoder."""
    with open(f"{OUT}/tuning_choice.json") as f:
        act = json.load(f)[setting].get("activation")
    return f" --activation {act}" if act else ""


def protocol(encoders, supervised, arch, extra=""):
    """The supervised baseline, then k-NN, the linear probe, analysis and fine-tuning of every encoder.
    arch: "cnn3" or "resnet18"."""
    sup, probe, ft = tuned(f"{arch}_supervised"), tuned(f"{arch}_linear_probe"), tuned(f"{arch}_finetune")
    runs = [f"--mode pretrain_supervise_encoder --label_fraction {f}{extra}{sup}" for f in FRACTIONS]
    supervised += activation(f"{arch}_supervised")  # its encoder is loaded with the activation it trained with
    for flags in (*encoders, supervised):
        for dataset in DATASETS:
            runs += [f"--mode knn {flags} --dataset {dataset}",
                     f"--mode linear_probe {flags} --dataset {dataset}{probe}",
                     f"--mode analysis {flags} --dataset {dataset}"]
    for flags in encoders:
        runs += [f"--mode finetune {flags} --dataset {dataset} --label_fraction {f}{ft}" for dataset in DATASETS for f in FRACTIONS]
    return runs


def run_all(runs):
    for i, flags in enumerate(runs, 1):
        print(f"\n=== {i}/{len(runs)}: {flags}")
        !python train.py {flags} --output_dir "{OUT}"


SMALL = ["--encoder dae", "--encoder dae --noise_level 0", "--encoder random"]
R = "--backbone resnet18"
RESNET = [f"--encoder dae {R} --decoder resnet18", f"--encoder dae --noise_level 0 {R} --decoder resnet18", f"--encoder random {R}"]

## Group 1: CNN-3
The DAE (salt and pepper, 50%) and the AE (`--noise_level 0`), 12 epochs each, then the full protocol.

In [ ]:
run_all(["--mode pretrain", "--mode pretrain --noise_level 0"] + protocol(SMALL, "--encoder supervised", "cnn3"))

## Group 2a: ResNet-18 pretraining
The DAE and the AE with the ResNet-18 decoder, 30 epochs: in 12 epochs their loss and k-NN were still improving (the kept checkpoint is still the one with the best val k-NN). The DAE with the light decoder, 12 epochs (its k-NN peaked at epoch 5), to compare the two decoders. Needs no tuning: start it at once.

In [ ]:
run_all([f"--mode pretrain {R} --decoder resnet18 --num_epochs 30",
         f"--mode pretrain --noise_level 0 {R} --decoder resnet18 --num_epochs 30",
         f"--mode pretrain {R}"])

## Group 2b: ResNet-18, full fine-tuning
The full protocol on the encoders of 2a, with the choices of `colab_tuning.ipynb`. Also the DAE with the light decoder by k-NN and the linear probe.

In [ ]:
run_all(protocol(RESNET, f"--encoder supervised {R}", "resnet18", extra=f" {R}")
        + [f"--mode knn --encoder dae {R}", f"--mode linear_probe --encoder dae {R}{tuned('resnet18_linear_probe')}"])

## Group 3: ResNet-18, partial fine-tuning
The encoders of group 2. conv1, bn1, layer1 and layer2 stay frozen; layer3, layer4, fc and the head train. Their folders have `finetune-from-layer3`.

In [ ]:
for name in ("salt_pepper_0.5", "salt_pepper_0.0"):
    path = f"{OUT}/encoders/dae_TinyImageNet_{name}_256_resnet18_resnet18dec_s42.pt"
    assert os.path.exists(path), f"group 2 has not trained {os.path.basename(path)} yet"
run_all([f"--mode finetune {flags} --dataset {dataset} --label_fraction {f} --finetune_from layer3{tuned('resnet18_finetune')}"
         for flags in RESNET for dataset in DATASETS for f in FRACTIONS])

## Group 4: seeds 1 and 2 of the small encoder
Group 1 again with another training seed (initialization, data order, noise). The splits and the label subsets stay the same. Every file and run folder ends in `_s1` or `_s2`.

In [ ]:
for seed in (1, 2):
    s = f" --seed {seed}"
    run_all([f"--mode pretrain{s}", f"--mode pretrain --noise_level 0{s}"]
            + [flags + s for flags in protocol(SMALL, "--encoder supervised", "cnn3")])

## Results
Every finished final run of this notebook, one row each, from its `run_info.txt`, in the order they started.

In [ ]:
import glob

import pandas as pd

rows = []
for path in glob.glob(f"{OUT}/date*_final_*/run_info.txt"):
    with open(path, encoding="utf-8") as f:
        rows.append(dict(line.rstrip("\n").split(": ", 1) for line in f if ": " in line))
runs = pd.DataFrame(rows).sort_values("started")
pd.set_option("display.max_rows", None, "display.max_colwidth", 120)
print(len(runs), "runs")
runs[["started", "folder", "epochs", "results", "training time"]]